# Day 5: Activation Functions (ReLU, Sigmoid, Softmax, GELU)

**Module 2 — Neural Network Basics | 100 Days of Data Science**

## Why This Matters
Without activation functions, stacking layers would be pointless — a chain of purely linear operations collapses into a single linear operation, no matter how many layers you add. Activation functions inject **non-linearity**, which is what lets an MLP actually learn complex patterns like XOR (Day 4).

Choosing the right activation function also directly affects how well gradients flow during training — a topic we'll revisit when we hit vanishing/exploding gradients.

## Topics Covered Today
1. Why non-linearity matters (proof by example)
2. Sigmoid
3. Tanh
4. ReLU and its variants (Leaky ReLU)
5. Softmax (recap + when to use it)
6. GELU (used in Transformers/BERT/GPT)
7. Side-by-side comparison
8. Practice exercises

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

print("NumPy version:", np.__version__)

---
## 1. Why Non-Linearity Matters

If every layer is purely linear (`z = Wx + b`), stacking layers is mathematically equivalent to a SINGLE linear layer — no matter how deep the network is.

In [ ]:
# Prove that stacking linear layers (no activation) collapses to one linear layer
np.random.seed(0)
x = np.random.rand(1, 3)

W1 = np.random.rand(3, 4)
W2 = np.random.rand(4, 2)

# Two "layers", no activation
out_two_layers = (x @ W1) @ W2

# Equivalent single combined layer
W_combined = W1 @ W2
out_single_layer = x @ W_combined

print("Output through 2 linear layers:\n", out_two_layers)
print("Output through 1 combined linear layer:\n", out_single_layer)
print("\nThey match! Without activation functions, depth adds NO extra representational power.")

---
## 2. Sigmoid

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Squashes input to range `(0, 1)`. Historically popular, but suffers from **vanishing gradients** at extreme values (the curve flattens, so gradients shrink to ~0).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

z = np.linspace(-10, 10, 200)

plt.figure(figsize=(6,4))
plt.plot(z, sigmoid(z), label='sigmoid(z)')
plt.plot(z, sigmoid_derivative(z), label="sigmoid'(z)")
plt.legend()
plt.title('Sigmoid and its Derivative')
plt.grid(True)
plt.show()

print("Max gradient value:", sigmoid_derivative(z).max(), "(occurs at z=0)")
print("Gradient at z=10:", sigmoid_derivative(np.array([10.0]))[0], "-- nearly zero, causes vanishing gradients")

---
## 3. Tanh

$$\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$$

Squashes input to range `(-1, 1)`. Zero-centered (unlike sigmoid), which usually helps optimization, but still suffers from vanishing gradients at extremes.

In [ ]:
def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z)**2

plt.figure(figsize=(6,4))
plt.plot(z, tanh(z), label='tanh(z)')
plt.plot(z, tanh_derivative(z), label="tanh'(z)")
plt.legend()
plt.title('Tanh and its Derivative')
plt.grid(True)
plt.show()

---
## 4. ReLU and Leaky ReLU

**ReLU** (Rectified Linear Unit): $\text{ReLU}(z) = \max(0, z)$

The default choice for hidden layers in most modern networks — cheap to compute, doesn't saturate for positive values, and trains faster in practice.

**Problem — "Dying ReLU":** if a neuron's input is always negative, its gradient is always 0, so it stops learning permanently.

**Leaky ReLU** fixes this by allowing a small non-zero gradient for negative inputs: $\text{LeakyReLU}(z) = \max(\alpha z, z)$

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_derivative(z, alpha=0.01):
    return np.where(z > 0, 1, alpha)

fig, axes = plt.subplots(1, 2, figsize=(12,4))

axes[0].plot(z, relu(z), label='ReLU(z)')
axes[0].plot(z, relu_derivative(z), label="ReLU'(z)")
axes[0].set_title('ReLU')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(z, leaky_relu(z), label='LeakyReLU(z)')
axes[1].plot(z, leaky_relu_derivative(z), label="LeakyReLU'(z)")
axes[1].set_title('Leaky ReLU')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

---
## 5. Softmax (Recap)

Covered in Day 3 — used ONLY on the **output layer** for multi-class classification, converting logits into a probability distribution that sums to 1. Never used in hidden layers.

In [ ]:
def softmax(logits):
    exp_scores = np.exp(logits - np.max(logits))
    return exp_scores / np.sum(exp_scores)

logits = np.array([2.0, 1.0, 0.1])
print("Softmax output:", softmax(logits), "| sums to:", softmax(logits).sum())

---
## 6. GELU (Gaussian Error Linear Unit)

$$\text{GELU}(z) = z \cdot \Phi(z)$$

where $\Phi(z)$ is the standard normal cumulative distribution function. A smoother alternative to ReLU — used in BERT, GPT, and most modern Transformer architectures. It weights inputs by how likely they are to be "kept" rather than a hard cutoff at 0.

In [ ]:
from scipy.stats import norm

def gelu(z):
    return z * norm.cdf(z)

plt.figure(figsize=(6,4))
plt.plot(z, relu(z), label='ReLU', linestyle='--')
plt.plot(z, gelu(z), label='GELU')
plt.legend()
plt.title('GELU vs ReLU -- GELU is smooth near zero')
plt.grid(True)
plt.show()

---
## 7. Side-by-Side Comparison

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(z, sigmoid(z), label='Sigmoid')
plt.plot(z, tanh(z), label='Tanh')
plt.plot(z, relu(z), label='ReLU')
plt.plot(z, leaky_relu(z), label='Leaky ReLU')
plt.plot(z, gelu(z), label='GELU')
plt.legend()
plt.title('All Activation Functions Compared')
plt.ylim(-2, 5)
plt.grid(True)
plt.show()

### Quick Reference: When to Use What

| Activation | Range | Common Use | Notes |
|---|---|---|---|
| Sigmoid | (0, 1) | Binary output layer | Vanishing gradient issue |
| Tanh | (-1, 1) | RNN hidden states (older) | Zero-centered, still saturates |
| ReLU | [0, ∞) | Default for hidden layers (CNNs, MLPs) | Fast, but can "die" |
| Leaky ReLU | (-∞, ∞) | Hidden layers, fixing dying ReLU | Small negative slope |
| Softmax | (0,1), sums to 1 | Multi-class output layer only | Never in hidden layers |
| GELU | (-≈0.17, ∞) | Transformers (BERT, GPT) | Smooth, state-of-the-art default |

---
## 8. Practice Exercises
Try these before Day 6:

1. Implement the ELU activation function ($\text{ELU}(z) = z$ if $z>0$ else $\alpha(e^z - 1)$) and plot it next to ReLU.
2. For `z = -5`, compute the derivative under sigmoid, tanh, and ReLU. Which is closest to zero (vanishing gradient)?
3. Modify the `SimpleMLP` from Day 4 to use `tanh` instead of `relu` in the hidden layer, and compare outputs.
4. Look up (or reason through) why Softmax should never be used in a hidden layer — what would happen to the network's expressive power?
5. In your own words: why is GELU preferred over ReLU in Transformer architectures?

In [ ]:
# Your practice code here


---
## Summary
- Without activation functions, depth is meaningless — stacked linear layers collapse into one
- **Sigmoid/Tanh** squash outputs but suffer from vanishing gradients at extremes
- **ReLU** is the standard default for hidden layers — fast, but can "die"
- **Leaky ReLU** fixes dying neurons with a small negative slope
- **Softmax** is reserved for multi-class output layers
- **GELU** is the modern default in Transformer architectures (BERT, GPT)

Next up: **Day 6 — Forward & Backward Propagation (implementing training from scratch)**

---
*Part of the 100 Days of Data Science series | DL-for-Data-Science repo*